# Fashion-MNIST Dataset - Autoencoder

### [Fashion-MNIST Dataset](https://github.com/zalandoresearch/fashion-mnist)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Han Xiao, Kashif Rasul, Roland Vollgraf, https://github.com/zalandoresearch/fashion-mnist

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy matplotlib scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
%matplotlib inline

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IMAGE_SIZE = 28
IMAGE_SIZE

In [ ]:
LATENT_DIM = 16
LATENT_DIM

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.001
LEARNING_RATE

In [ ]:
EPOCHS = 10
EPOCHS

In [ ]:
BATCH_SIZE = 256
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Load Dataset

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.FashionMNIST('data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
len(train_data), len(test_data)

## Create Model

The encoder compresses each 784-pixel image to a 16-number latent vector. The decoder reconstructs the image from that vector.

In [ ]:
class Autoencoder(nn.Module):
    """
    A fully connected autoencoder for 28x28 grayscale images.
    """

    def __init__(self, latent_dim=LATENT_DIM):
        """
        Initialize the encoder and decoder.

        Parameters:
            latent_dim (int): Size of the latent vector.

        Returns:
            None
        """
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, IMAGE_SIZE * IMAGE_SIZE),
            nn.Sigmoid())

    def forward(self, x):
        """
        Encode and reconstruct a batch of images.

        Parameters:
            x (torch.Tensor): Batch of images.

        Returns:
            tuple: Reconstruction and latent vector.
        """
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction, latent

## Instantiate Model, Loss, and Optimizer

In [ ]:
torch.manual_seed(SEED)
model = Autoencoder().to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the autoencoder for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Reconstruction loss.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss.
    """
    model.train()
    total = 0.0
    for x, _ in loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()
        reconstruction, _ = model(x)
        loss = loss_fn(reconstruction, x.view(x.size(0), -1))
        loss.backward()
        optimizer.step()
        total += loss.item() * len(x)
    return total / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    """
    Evaluate the autoencoder over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.
        loss_fn (nn.Module): Reconstruction loss.

    Returns:
        float: Mean reconstruction loss.
    """
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(DEVICE)
            reconstruction, _ = model(x)
            total += loss_fn(reconstruction, x.view(x.size(0), -1)).item() * len(x)
    return total / len(loader.dataset)

### Training Loop

In [ ]:
history = []
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    test_loss = evaluate(model, test_loader, loss_fn)
    history.append(test_loss)
    print(f"Epoch {epoch + 1:2d} | train loss {train_loss:.4f} | test loss {test_loss:.4f}")

### Visualize Training

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction loss')
plt.title('Autoencoder training')
plt.tight_layout()
plt.show()

## Visualize Reconstructions

### Function

In [ ]:
def show_reconstructions(model, dataset, n=8):
    """
    Plot original images above their reconstructions.

    Parameters:
        model (nn.Module): Trained autoencoder.
        dataset (Dataset): Source of images.
        n (int): Number of images to show.

    Returns:
        None
    """
    model.eval()
    images = torch.stack([dataset[i][0] for i in range(n)]).to(DEVICE)
    with torch.no_grad():
        reconstruction, _ = model(images)
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].imshow(reconstruction[i].cpu().view(28, 28), cmap='gray')
        axes[1, i].axis('off')
    fig.tight_layout()
    plt.show()

### Run

In [ ]:
show_reconstructions(model, test_data, n=8)

## Explore the Latent Space

Project the 16-dimensional latent vectors down to two dimensions with PCA and color by the true class. Similar garments cluster together.

In [ ]:
model.eval()
latents = []
labels = []
with torch.no_grad():
    for x, y in test_loader:
        _, latent = model(x.to(DEVICE))
        latents.append(latent.cpu().numpy())
        labels.append(y.numpy())
latents = np.concatenate(latents)
labels = np.concatenate(labels)
coords = PCA(n_components=2).fit_transform(latents)
plt.figure(figsize=(8, 8))
plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10', s=4)
plt.title('Latent space colored by class')
plt.colorbar()
plt.tight_layout()
plt.show()

## Anomaly Detection

The reconstruction error is a score for how unusual an image is. Images the model has not learned to reconstruct have high error.

In [ ]:
def reconstruction_errors(model, dataset, n=1000):
    """
    Compute per-image reconstruction errors.

    Parameters:
        model (nn.Module): Trained autoencoder.
        dataset (Dataset): Source of images.
        n (int): Number of images to score.

    Returns:
        numpy.ndarray: Reconstruction errors.
    """
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(n):
            image, _ = dataset[i]
            image = image.to(DEVICE).unsqueeze(0)
            reconstruction, _ = model(image)
            error = torch.mean((reconstruction - image.view(1, -1)) ** 2).item()
            errors.append(error)
    return np.array(errors)

In [ ]:
errors = reconstruction_errors(model, test_data, n=1000)
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=50)
plt.xlabel('Reconstruction error')
plt.ylabel('Count')
plt.title('Reconstruction error distribution')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
torch.save(model.state_dict(), 'autoencoder_fashion_mnist.pt')
print('saved autoencoder_fashion_mnist.pt')

## Load Model

In [ ]:
loaded_model = Autoencoder().to(DEVICE)
loaded_model.load_state_dict(torch.load('autoencoder_fashion_mnist.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded autoencoder_fashion_mnist.pt')

## Inference

### Function

In [ ]:
def reconstruct(model, image):
    """
    Reconstruct one image and return its error.

    Parameters:
        model (nn.Module): Trained autoencoder.
        image (torch.Tensor): A 1x28x28 image tensor.

    Returns:
        tuple: Reconstructed image and error.
    """
    model.eval()
    with torch.no_grad():
        batch = image.unsqueeze(0).to(DEVICE)
        reconstruction, _ = model(batch)
        error = torch.mean((reconstruction - batch.view(1, -1)) ** 2).item()
    return reconstruction.view(28, 28).cpu(), error

### Run Inference

In [ ]:
image, label = test_data[5]
reconstruction, error = reconstruct(loaded_model, image)
print(f"Label: {label} | Reconstruction error: {error:.4f}")
plt.imshow(reconstruction, cmap='gray')
plt.title('Reconstruction')
plt.axis('off')
plt.show()